### **Import Library**

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.naive_bayes import MultinomialNB, CategoricalNB
from sklearn.metrics import accuracy_score

### Read dataset

In [5]:
df = pd.read_csv("../Data/diabetes_70k.csv")

# Separate features and target
X = df.drop(columns=["class"])
y = df["class"]

print("Number of rows:", X.shape[0])
print("Number of features:", X.shape[1])
print("Class distribution:")
print(y.value_counts())

Number of rows: 70692
Number of features: 21
Class distribution:
class
0    35346
1    35346
Name: count, dtype: int64


**Task 1 — OneHotEncoder vs OrdinalEncoder with Naive Bayes**  
Using scikit-learn, evaluate the following encoder–classifier combinations:  
• OneHotEncoder → MultinomialNB  
• OrdinalEncoder → CategoricalNB  
For this task:
1. Apply each encoder to the dataset
2. Use a 50:50 train–test split.
3. Train the classifier.
4. Repeat the experiment multiple times (e.g., 20 runs) using different random seeds.
5. Report the average accuracy for each encoder.

##### OneHotEncoder with MultinomialNB

In [6]:
def run_onehot_multinomial(X_train, X_test, y_train, y_test):
    # Categories unseen in the training set are all 0
    ohe = OneHotEncoder(handle_unknown="ignore")
    X_train_enc = ohe.fit_transform(X_train)
    X_test_enc = ohe.transform(X_test)

    model = MultinomialNB()
    model.fit(X_train_enc, y_train)

    return accuracy_score(y_test, model.predict(X_test_enc))

##### OrdinalEncoder with CategoricalNB

In [7]:
def run_ordinal_categorical(X_train, X_test, y_train, y_test):
    # Unseen categories in the test set are first mark as -1
    ordinal = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    X_train_enc = ordinal.fit_transform(X_train)
    X_test_enc = ordinal.transform(X_test)

    # Number of categories per column
    n_cats = np.array([len(c) for c in ordinal.categories_])

    # Move unseen value to extra index
    X_test_enc = np.where(X_test_enc == -1, n_cats, X_test_enc).astype(int)
    X_train_enc = X_train_enc.astype(int)

    model = CategoricalNB(min_categories=n_cats + 1)
    model.fit(X_train_enc, y_train)

    return accuracy_score(y_test, model.predict(X_test_enc))

##### Repeat the experiment 20 times with 50:50

In [8]:
n_runs = 20
ohe_acc = []
ord_acc = []

for seed in range(n_runs):
    # 50:50 split with a different rand seed
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.5, random_state=seed
    )

    acc_ohe = run_onehot_multinomial(X_train, X_test, y_train, y_test)
    acc_ord = run_ordinal_categorical(X_train, X_test, y_train, y_test)

    ohe_acc.append(acc_ohe)
    ord_acc.append(acc_ord)

    print(f"Run {seed + 1}")
    print(f"OneHotEncoder + MultinomialNB accuracy: {acc_ohe:.4f}")
    print(f"OrdinalEncoder + CategoricalNB accuracy: {acc_ord:.4f}")
    print()

Run 1
OneHotEncoder + MultinomialNB accuracy: 0.7365
OrdinalEncoder + CategoricalNB accuracy: 0.7365

Run 2
OneHotEncoder + MultinomialNB accuracy: 0.7342
OrdinalEncoder + CategoricalNB accuracy: 0.7342

Run 3
OneHotEncoder + MultinomialNB accuracy: 0.7370
OrdinalEncoder + CategoricalNB accuracy: 0.7370

Run 4
OneHotEncoder + MultinomialNB accuracy: 0.7297
OrdinalEncoder + CategoricalNB accuracy: 0.7297

Run 5
OneHotEncoder + MultinomialNB accuracy: 0.7334
OrdinalEncoder + CategoricalNB accuracy: 0.7334

Run 6
OneHotEncoder + MultinomialNB accuracy: 0.7331
OrdinalEncoder + CategoricalNB accuracy: 0.7331

Run 7
OneHotEncoder + MultinomialNB accuracy: 0.7345
OrdinalEncoder + CategoricalNB accuracy: 0.7345

Run 8
OneHotEncoder + MultinomialNB accuracy: 0.7339
OrdinalEncoder + CategoricalNB accuracy: 0.7339

Run 9
OneHotEncoder + MultinomialNB accuracy: 0.7330
OrdinalEncoder + CategoricalNB accuracy: 0.7330

Run 10
OneHotEncoder + MultinomialNB accuracy: 0.7324
OrdinalEncoder + Categorical

##### The avg accuracy

In [9]:
print("Average accuracy over", n_runs, "runs")
print(f"OneHotEncoder + MultinomialNB: {np.mean(ohe_acc):.4f}")
print(f"OrdinalEncoder + CategoricalNB: {np.mean(ord_acc):.4f}")

Average accuracy over 20 runs
OneHotEncoder + MultinomialNB: 0.7333
OrdinalEncoder + CategoricalNB: 0.7333
